[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
def causal_attention(Q, K, V):
  B, s, d= Q.shape

  W_q= torch.nn.Parameter(torch.randn(Q.shape[-1], Q.shape[-1]))
  W_k= torch.nn.Parameter(torch.randn(K.shape[-1], K.shape[-1]))
  W_v= torch.nn.Parameter(torch.randn(V.shape[-1], V.shape[-1]))

  attn_score= torch.matmul(Q, K.transpose(-2, -1))
  mask= torch.triu(torch.ones(s, s), diagonal= 1).bool()
  attn_score= attn_score.masked_fill(mask, float('-inf'))
  attn_score= torch.softmax(attn_score/math.sqrt(d), dim= -1)
  attn_score= torch.matmul(attn_score, V)

  return attn_score

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

In [ ]:
from torch_judge import check
check('causal_attention')